In [ ]:
import sys, os
import matplotlib.pyplot as plt

# Project styling
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'src'))
from dimp.utils import init_matplotlib, get_colors
init_matplotlib()
colors = get_colors()

# Import plotting functions from the visualization script
from plot_pann_clqr_dt import (
    load_all_results,
    plot_method_results,
    plot_hs_comparison,
    print_continuous_costs,
    plot_baseline_sweep,
    _build_method_solutions,
    plot_density_analysis,
    plot_cross_correlation_analysis,
)

In [ ]:
data_dir = os.path.join("data", "pann_clqr_dt")
n = 160
n_zoh3 = 80
all_results = load_all_results(data_dir, n=n, n_zoh3=n_zoh3)
print(f"Loaded methods: {list(all_results.keys())}")

## Example Pannocchia 🌽

Contnuous LTI system with dynamics
$$
\dot{s} = A s + B u
$$

3 states and 1 input.

### Define the problem matrices, the initial state, the time horizon, and the LQR matrices.

### Create the Optimization Problem

Classic constrained LQR problem.
Discretized with first-order Euler.
$$
\begin{align*}
& \min_{\substack{s_{k+1}, u_k \\ k=0, \dots, N}} \quad & \sum_{k=0}^{N} \left( s_k^T Q s_k + u_k^T R u_k \right), \\
& \text{s.t.} \quad & s_{k+1} = s_k + \Delta t (A s_k + B u_k), \\
& & s_0 = s_{\text{init}}, \\
& & u_k \in U.
\end{align*}
$$

### Study how the solution changes with different number of samples.

With $n=80$ samples the system is unstable.
From $n=160$ timesteps the OCP stabilizes the system.
However, increasing the number of timesteps further improves the solution (see the optimal cost value).

In [ ]:
plot_baseline_sweep(results_dir=None, show=True)

## DQP with Auxiliary Variables

### Create the Parametrized CLQR Problem

Discretize then linearize the (nonlinear) dynamics.
$$
s_{k+1} = \bar{s}_k + \bar{\delta}_k (A \bar{s}_k + B u_k) + (I + \bar{\delta}_k A) \tilde{s}_k + \bar{\delta}_k B u_k + (A \bar{s}_k + B u_k) \tilde{\delta}_k,
$$
where $\tilde{\square} = \square - \bar{\square}$ and $\delta_k$ is an auxiliary variable that represents the time step lengths.

The optimization vector of the QP is $\begin{bmatrix} \tilde{s}_{k=1,\dots,N} & \tilde{u}_{k=0,\dots,N-1} & \tilde{\delta}_{k=0,\dots,N} -1\end{bmatrix}$.

The QP is parametrized by the parameter vector $\bar{\delta} = \begin{bmatrix} \bar{\delta}_1 & \dots & \bar{\delta}_N \end{bmatrix}^T$.
$$
\begin{align*}
& \min_{\substack{s_{k+1}, u_k, \delta_k \\ k=0, \dots, N}} \quad & \sum_{k=0}^{N} \left(\bar{\delta}_k \left( s_k^T Q s_k + u_k^T R u_k \right) + w_\delta \tilde{\delta}_k^2 \right), \\
& \text{s.t.} \quad & s_{k+1} = \bar{s}_k + \bar{\delta}_k (A \bar{s}_k + B \bar{u}_k) + (I + \bar{\delta}_k A) \tilde{s}_k + \bar{\delta}_k B \tilde{u}_k + (A \bar{s}_k + B \bar{u}_k) \tilde{\delta}_k, \\
& & s_0 = s_{\text{init}}, \\
& & u_k \in U.
\end{align*}
$$

### Task Loss

| Method          | Loss                                                                                                           |
| --------------- | -------------------------------------------------------------------------------------------------------------- |
| Unscaled        | $$\mathcal{L}_1 = \sum_{k=0}^{N} \left(\lVert s_k \rVert^2_Q  + \lVert u_k \rVert^2_R \right)$$                |
| Time scaled     | $$\mathcal{L}_2 = \sum_{k=0}^{N} \delta_k \left(\lVert s_k \rVert^2_Q  + \lVert u_k \rVert^2_R \right)$$       |
| Time bar scaled | $$\mathcal{L}_3 = \sum_{k=0}^{N} \bar{\delta}_k \left(\lVert s_k \rVert^2_Q  + \lVert u_k \rVert^2_R \right)$$ |

### Training Loop

In [ ]:
plot_method_results("aux", all_results["aux"], results_dir=None, show=True)

## DQP with Reparametrized Parameters

### DQP With Reparametrized Timesteps

Reparametrize time steps with the simplex:
$$
\Delta t_k = \epsilon + (T - n \epsilon) \frac{e^{\theta_k}}{\sum_j e^{\theta_j}}
$$
This enforces both positivity and the total time constraint $\sum_k \Delta t_k = T$ without discontinuous updates.

The optimization vector of the QP is $\begin{bmatrix} \tilde{s}_{k=1,\dots,N} & \tilde{u}_{k=0,\dots,N-1}\end{bmatrix}$.

$$
\begin{align*}
& \min_{\substack{s_{k+1}, u_k \\ k=0, \dots, N}} \quad & \sum_{k=0}^{N} \left(\Delta t_k \left( s_k^T Q s_k + u_k^T R u_k \right)\right), \\
& \text{s.t.} \quad & s_{k+1} = \bar{s}_k + \Delta t_k (A \bar{s}_k + B \bar{u}_k) + (I + \Delta t_k A) \tilde{s}_k + \Delta t_k B \tilde{u}_k, \\
& & s_0 = s_{\text{init}}, \\
& & u_k \in U.
\end{align*}
$$

The OCP uses $\Delta t_k (\theta)$ as a parameter. The optimizer optimizes $\theta$. Gradients do flow through the softmax function in PyTorch.

### Task Loss

| Method       | Loss                                                                                                       |
| ------------ | ---------------------------------------------------------------------------------------------------------- |
| ~~Unscaled~~ | $$\mathcal{L}_1 = \sum_{k=0}^{N} \left(\lVert s_k \rVert^2_Q  + \lVert u_k \rVert^2_R \right)$$            |
| Time scaled  | $$\mathcal{L}_2 = \sum_{k=0}^{N} \Delta t_k \left(\lVert s_k \rVert^2_Q  + \lVert u_k \rVert^2_R \right)$$ |

In [ ]:
plot_method_results("rep", all_results["rep"], results_dir=None, show=True)

## Loss Hyper Sampling

### Training with Hyper-Sampling (HS) Losses

Problem that HS attempts to solve: the loss is computed only along the trajectory (non-uniform) sampling points. The loss may not reflect the true cost over the entire time horizon.

A possible solution would be to compute the exact cost along the trajectory assuming ZOH inputs. Instead, here, we propose two different strategies.

#### Dense Uniform Grid

The non-uniform grid is replaced with a dense uniform grid. The input is constant within the original non-uniform intervals. The dynamics is integrated with first-order Euler on the dense grid.

**Main problem**: the dense grid is computed with `detached()`, the gradients do not flow through it.

#### Substeps

Within each original non-uniform interval, we introduce $M$ substeps. The dynamics is integrated with first-order Euler on the substeps.

In [ ]:
plot_method_results("hs", all_results["hs"], results_dir=None, show=True)

In [ ]:
plot_hs_comparison(all_results["hs"], n, results_dir=None, show=True)

In [ ]:
print_continuous_costs(all_results, n)

## DQP With ZOH Exact Discretization

Use the exact discretization of the LTI system with zero-order hold (ZOH).

OCP parameters:
$$
A_{d, k}, B_{d, k}
= \operatorname{ZOH}(A, B, \Delta t_k)
= \exp \left( \begin{bmatrix} A & B \\ 0 & 0 \end{bmatrix} \Delta t_k \right)
= \begin{bmatrix} A_{d, k} & B_{d, k} \\ 0 & I \end{bmatrix}
$$

### No Cost Scaling With Interval Duration

In [ ]:
plot_method_results("zoh", all_results["zoh"], results_dir=None, show=True)

### With Cost Scaling With Interval Duration

In [ ]:
plot_method_results("zoh2", all_results["zoh2"], results_dir=None, show=True)

## DQP with Exact ZOH Discretization and Exact Integrated Cost

This section implements the approach from Pannocchia et al.: **exact ZOH discretization** of both dynamics and quadratic stage cost, with all nonlinear dependence on $\Delta t_k$ computed outside the QP.

### Exact ZOH Integrated Stage Cost

Under ZOH, the state evolves on $[t_k, t_{k+1}]$ as $x(\tau) = e^{A\tau} x_k + \Gamma(\tau) u_k$ where $\Gamma(\tau) = \int_0^\tau e^{As} B\, ds$.
The exact integrated cost over interval $k$ is:
$$\ell_k = \int_0^{\Delta t_k} \bigl(x(\tau)^T Q\, x(\tau) + u_k^T R\, u_k\bigr)\, d\tau = z_k^T \bar{W}_k z_k$$

where $z_k = \begin{bmatrix} x_k \\ u_k \end{bmatrix}$ and $\bar{W}_k = \begin{bmatrix} Q_{d,k} & M_{d,k} \\ M_{d,k}^T & R_{d,k} \end{bmatrix} \succeq 0$ with:
- $Q_{d,k} = \int_0^{\Delta t_k} e^{A^T\tau} Q\, e^{A\tau}\, d\tau$ (observability-like Gramian)
- $M_{d,k} = \int_0^{\Delta t_k} e^{A^T\tau} Q\, \Gamma(\tau)\, d\tau$ (cross term)
- $R_{d,k} = \int_0^{\Delta t_k} \Gamma(\tau)^T Q\, \Gamma(\tau)\, d\tau + \Delta t_k R$

### Van Loan Block Matrix Exponential

Define $\hat{A} = \begin{bmatrix} A & B \\ 0 & 0 \end{bmatrix}$ and $\hat{Q} = \begin{bmatrix} Q & 0 \\ 0 & 0 \end{bmatrix}$. The cost-related integrals form the observability Gramian $W^Q = \int_0^{\Delta t} e^{\hat{A}^T \tau} \hat{Q}\, e^{\hat{A}\tau}\, d\tau$, extracted via:
$$\exp\left(\begin{bmatrix} -\hat{A}^T & \hat{Q} \\ 0 & \hat{A} \end{bmatrix} \Delta t\right) = \begin{bmatrix} \star & e^{-\hat{A}^T \Delta t}\, W^Q \\ 0 & e^{\hat{A}\Delta t} \end{bmatrix}$$

From the bottom-right block we also read $A_d$ and $B_d$. The full cost matrix is $\bar{W} = W^Q + \text{diag}(0, \Delta t\, R)$.

### Why This Stays a QP and Parameter-Affine

The matrices $(\bar{W}_k, A_{d,k}, B_{d,k})$ depend nonlinearly on $\Delta t_k$ through `torch.matrix_exp`, but they are computed **outside** the QP. Inside the QP, they are fixed parameters:
$$\min_{x,u}\;\sum_k \|L_k^T z_k\|^2 \quad \text{s.t. } x_{k+1} = A_{d,k} x_k + B_{d,k} u_k,\; |u_k| \le u_{\max}$$

where $L_k = \text{chol}(\bar{W}_k)$ is a parameter matrix. The expression $L_k^T z_k$ is **affine in both parameters and variables**, so `cp.sum_squares(...)` is DPP-compliant in CVXPY. All nonlinearity in $\Delta t$ resides in the parameter computation, preserving the QP structure.

**Note on terminal cost:** No terminal cost $x_N^T P x_N$ is included, consistent with the other implementations in this notebook.

In [ ]:
plot_method_results("zoh3", all_results["zoh3"], results_dir=None, show=True)

## Sampling Density and Trajectory Change Analysis

### Sampling Density vs Trajectory Changes

In [ ]:
method_solutions = _build_method_solutions(all_results, n)
plot_density_analysis(method_solutions, colors, results_dir=None, show=True)

### Cross-Correlation (Time-Lagged)

Cross correlation of the Sampling Density (SD) with:
- $\| \Delta u \|$
- $\| \Delta s \|_2$
- $\| u \|$
- $\| s \|_2$

On the y-axis, the cross-correlation factor.

The box indicates the maximum CC and the time lag at which it happens.

The dotted lines at $\pm 1.96 / \sqrt{n}$ ≈ ±0.155 (for n=160) are the 95% CI. Outside bounds -> statistically significant correlation.

- **lag = 0**: Instantaneous correlation
- **lag > 0**: Does high sampling density *precede* large changes?
- **lag < 0**: Does high sampling density *follow* large changes?

In [ ]:
plot_cross_correlation_analysis(method_solutions, colors, results_dir=None, show=True)